In [ ]:
import os
from openai import OpenAI
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])



# P3 Reasoning Metrics Evaluation

This notebook evaluates reason-level metrics for model outputs against ground-truth data.
It computes two metric groups:
- Group 1: strict NLI-based metrics (using the same CrossEncoder NLI setup from P2/NLI.ipynb)
- Group 2: continuous cross-encoder overlap metrics (BAAI/bge-reranker-v2-m3)

Outputs:
- Per-example metrics table (preview + CSV text)
- Aggregate summary table (preview + CSV text)
- Pairwise reason scores (preview + CSV text)

Notes:
- This notebook reuses the NLI implementation pattern from `V1/P2/NLI.ipynb` — same model, label order, and softmax mapping.
- Future TODOs: verdict-level scoring, hierarchical reasoning scoring, composite final score.


In [ ]:

# Imports
import os
import json
import math
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sentence_transformers import CrossEncoder
from scipy.optimize import linear_sum_assignment


In [ ]:

# Config
BASE_DIR = Path.cwd()
# Paths are relative to this notebook's folder (V1/P3/testing)
DATASET_PATH = (BASE_DIR / "../data/processed/dataset.json").resolve()   
PREDICTIONS_PATH = (BASE_DIR / "../outputs/qwen25_05b_lora_sft_top3_epoch4_predictions.json").resolve()  # combined file supported
P2_NLI_NOTEBOOK_PATH = Path("/Users/nishchayjaiswal/Documents/Development/Verdict POC/V1/P2/NLI.ipynb")
COMBINED_INPUT = True  # If True, expect a single file with gt_* and pred_* fields

BATCH_SIZE = 16
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Using device: {DEVICE}")
print(f"Dataset path: {DATASET_PATH}")
print(f"Predictions path: {PREDICTIONS_PATH}")
print(f"Reused NLI notebook: {P2_NLI_NOTEBOOK_PATH}")


## Utility Functions
- Robust loaders and normalizers for ground-truth and predictions JSON
- Alignment by `id`


In [ ]:

# Utility helpers

def load_json(path: Path):
    with open(path, 'r') as f:
        return json.load(f)


def _as_list(x):
    if x is None:
        return []
    if isinstance(x, list):
        return x
    if isinstance(x, str):
        return [x]
    # Convert other types to str and keep
    return [str(x)]


def normalize_reasons(reasons):
    out = []
    for r in _as_list(reasons):
        if r is None:
            continue
        if isinstance(r, str):
            s = r.strip()
        else:
            try:
                s = str(r)
            except Exception:
                s = json.dumps(r, ensure_ascii=False)
        if s:
            out.append(s)
    return out


def normalize_example(ex: dict, source_name: str = "unknown") -> dict | None:
    if not isinstance(ex, dict):
        warnings.warn(f"Skipping non-dict example from {source_name}: {type(ex)}")
        return None
    id_ = ex.get('id') or ex.get('uid') or ex.get('qid')
    if not id_:
        warnings.warn(f"Skipping example without id from {source_name}")
        return None
    verdict = ex.get('verdict')
    reasons = normalize_reasons(ex.get('reasons'))
    return {
        'id': str(id_),
        'verdict': verdict if verdict is None or isinstance(verdict, str) else str(verdict),
        'reasons': reasons,
    }


def normalize_dataset(obj, source_name: str) -> list[dict]:
    items = []
    if isinstance(obj, dict):
        # dict keyed by id
        for k, v in obj.items():
            if isinstance(v, dict):
                v = {**v, 'id': v.get('id', k)}
            else:
                v = {'id': k, 'reasons': v}
            ne = normalize_example(v, source_name)
            if ne:
                items.append(ne)
    elif isinstance(obj, list):
        for v in obj:
            ne = normalize_example(v, source_name)
            if ne:
                items.append(ne)
    else:
        warnings.warn(f"Unsupported JSON top-level type for {source_name}: {type(obj)}")
    # Ensure unique ids (last wins)
    dedup = {}
    for ex in items:
        dedup[ex['id']] = ex
    return list(dedup.values())


def build_id_map(examples: list[dict]) -> dict[str, dict]:
    return {ex['id']: ex for ex in examples}


In [ ]:

# Split a combined predictions file (with gt_* and pred_* fields) into GT and Pred lists

def split_combined_predictions(obj) -> tuple[list[dict], list[dict]]:
    items = []
    if isinstance(obj, list):
        items = obj
    elif isinstance(obj, dict):
        # if stored as dict keyed by id
        items = [{**v, 'id': k} if isinstance(v, dict) else {'id': k} for k, v in obj.items()]
    else:
        return [], []

    gt_list, pred_list = [], []
    for it in items:
        id_ = str(it.get('id')) if it.get('id') is not None else None
        if not id_:
            continue
        # Ground truth fields
        gt_verdict = it.get('gt_verdict') or it.get('verdict')
        gt_reasons = it.get('gt_reasons') or it.get('reasons')
        gt_ex = normalize_example({'id': id_, 'verdict': gt_verdict, 'reasons': gt_reasons}, 'combined_gt')
        if gt_ex:
            gt_list.append(gt_ex)
        # Prediction fields
        pred_verdict = it.get('pred_verdict') or it.get('prediction')
        pred_reasons = it.get('pred_reasons') or it.get('pred_reasons_list')
        pred_ex = normalize_example({'id': id_, 'verdict': pred_verdict, 'reasons': pred_reasons}, 'combined_pred')
        if pred_ex:
            pred_list.append(pred_ex)
    return gt_list, pred_list



## Reused NLI Code (from P2)
This section mirrors the NLI setup and behavior from `V1/P2/NLI.ipynb`:
- Model: `cross-encoder/nli-deberta-v3-base` via `sentence_transformers.CrossEncoder`
- Label order: `["contradiction", "entailment", "neutral"]`
- Probabilities: softmax over raw scores; indices preserved as above
- Batch inference using `.predict(pairs, batch_size=...)`

We keep behavior identical and simply wrap it in a small helper class for reuse.


In [ ]:

# NLI wrapper (behavior preserved from P2)
class NLIPredictor:
    def __init__(self, model_name: str = 'cross-encoder/nli-deberta-v3-base', device: str = None):
        if device is None:
            device = 'cuda' if torch.cuda.is_available() else 'cpu'
        self.device = device
        # CrossEncoder supports passing device
        self.model = CrossEncoder(model_name, device=self.device)
        # Label mapping/order from P2
        self.label_mapping = ['contradiction', 'entailment', 'neutral']
        self.idx = {
            'contradiction': 0,
            'entailment': 1,
            'neutral': 2,
        }

    @torch.no_grad()
    def predict(self, pairs: list[tuple[str, str]], batch_size: int = 16) -> dict:
        if not pairs:
            empty = np.zeros((0, 3), dtype=np.float32)
            return {
                'logits': empty,
                'probs': empty,
                'labels_idx': np.array([], dtype=np.int64),
                'labels_str': [],
            }
        # CrossEncoder returns raw scores with shape [N, 3]
        scores = self.model.predict(pairs, batch_size=batch_size)
        # Softmax to probabilities
        t = torch.tensor(scores, dtype=torch.float32)
        probs = torch.softmax(t, dim=1).cpu().numpy()
        logits = t.cpu().numpy()
        labels_idx = probs.argmax(axis=1)
        labels_str = [self.label_mapping[i] for i in labels_idx]
        return {
            'logits': logits,
            'probs': probs,
            'labels_idx': labels_idx,
            'labels_str': labels_str,
        }



## Cross-Encoder (BAAI/bge-reranker-v2-m3)
Continuous overlap scoring used for similarity matrices and matching.


In [ ]:

class ReasoningReranker:
    def __init__(self, model_name: str = 'BAAI/bge-reranker-v2-m3', device: str = None, max_length: int = 512):
        if device is None:
            device = 'cuda' if torch.cuda.is_available() else 'cpu'
        self.device = device
        self.model_name = model_name
        self.max_length = max_length
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name)
        self.model.to(self.device)
        self.model.eval()

    @torch.no_grad()
    def score_pairs(self, pairs: list[tuple[str, str]], batch_size: int = 16) -> dict:
        if not pairs:
            empty = np.zeros((0,), dtype=np.float32)
            return {
                'raw': empty,
                'norm': empty,
            }
        raw_scores = []
        for i in range(0, len(pairs), batch_size):
            chunk = pairs[i:i+batch_size]
            a_list = [a for a, _ in chunk]
            b_list = [b for _, b in chunk]
            inputs = self.tokenizer(
                a_list,
                b_list,
                padding=True,
                truncation=True,
                max_length=self.max_length,
                return_tensors='pt'
            )
            inputs = {k: v.to(self.device) for k, v in inputs.items()}
            logits = self.model(**inputs).logits  # [bs, 1]
            raw = logits.squeeze(-1).detach().cpu().float().numpy()
            raw_scores.append(raw)
        raw_scores = np.concatenate(raw_scores, axis=0) if raw_scores else np.zeros((0,), dtype=np.float32)
        norm_scores = 1 / (1 + np.exp(-raw_scores))  # sigmoid
        return {
            'raw': raw_scores,
            'norm': norm_scores,
        }



## Pairwise Scoring
Build pairwise NLI and cross-encoder matrices for GT reasons × Predicted reasons.


In [ ]:

# Pairwise scoring utilities

def compute_nli_matrix(gt_reasons: list[str], pred_reasons: list[str], nli: NLIPredictor, batch_size: int = 16):
    G, P = len(gt_reasons), len(pred_reasons)
    if G == 0 or P == 0:
        shape = (G, P)
        nan = np.full(shape, np.nan, dtype=np.float32)
        empty_labels = np.empty(shape, dtype=object)
        return {
            'contradiction_prob': nan,
            'entailment_prob': nan,
            'neutral_prob': nan,
            'labels_idx': np.full(shape, -1, dtype=np.int64),
            'labels_str': empty_labels,
            'logits': np.full((*shape, 3), np.nan, dtype=np.float32),
        }
    pairs = [(g, p) for g in gt_reasons for p in pred_reasons]
    pred = nli.predict(pairs, batch_size=batch_size)
    probs = pred['probs']  # [G*P, 3]
    logits = pred['logits']
    # reshape
    probs = probs.reshape(G, P, 3)
    logits = logits.reshape(G, P, 3)
    contradiction = probs[:, :, nli.idx['contradiction']]
    entailment = probs[:, :, nli.idx['entailment']]
    neutral = probs[:, :, nli.idx['neutral']]

    # labels idx/str per pair
    labels_idx = probs.argmax(axis=2)
    # labels_str matrix
    label_map = ['contradiction', 'entailment', 'neutral']
    labels_str = np.empty((G, P), dtype=object)
    for i in range(G):
        for j in range(P):
            idx = int(labels_idx[i, j]) if not np.isnan(labels_idx[i, j]) else -1
            labels_str[i, j] = label_map[idx] if 0 <= idx < 3 else ''

    return {
        'contradiction_prob': contradiction,
        'entailment_prob': entailment,
        'neutral_prob': neutral,
        'labels_idx': labels_idx,
        'labels_str': labels_str,
        'logits': logits,
    }


def compute_reranker_matrix(gt_reasons: list[str], pred_reasons: list[str], reranker: ReasoningReranker, batch_size: int = 16):
    G, P = len(gt_reasons), len(pred_reasons)
    if G == 0 or P == 0:
        shape = (G, P)
        return {
            'raw': np.full(shape, np.nan, dtype=np.float32),
            'norm': np.full(shape, np.nan, dtype=np.float32),
        }
    pairs = [(g, p) for g in gt_reasons for p in pred_reasons]
    res = reranker.score_pairs(pairs, batch_size=batch_size)
    raw = res['raw'].reshape(G, P)
    norm = res['norm'].reshape(G, P)
    return {'raw': raw, 'norm': norm}



## Matching (Hungarian)
Similarity comes from normalized reranker scores. Cost = 1 - similarity.
We allow rectangular matrices and match up to the smaller side.


In [ ]:

# Hungarian matching

def hungarian_match_from_scores(score_matrix: np.ndarray):
    G, P = score_matrix.shape
    if G == 0 or P == 0:
        return np.array([], dtype=int), np.array([], dtype=int), np.array([], dtype=float)
    # Replace NaNs with 0 similarity to avoid issues
    sim = np.nan_to_num(score_matrix, nan=0.0, posinf=0.0, neginf=0.0)
    cost = 1.0 - sim
    rows, cols = linear_sum_assignment(cost)
    matched_scores = sim[rows, cols]
    return rows, cols, matched_scores



## Metric Functions
Group 1 (NLI-based) and Group 2 (cross-encoder-based).
Edge cases are handled conservatively with NaNs where denominators are 0.


In [ ]:

# Metric computations

EPS = 1e-12


def safe_mean(arr):
    arr = np.array(arr, dtype=float)
    if arr.size == 0:
        return np.nan
    # ignore NaNs
    m = np.nanmean(arr)
    return float(m) if not np.isnan(m) else np.nan


def compute_group1_metrics(nli: dict, match_rows: np.ndarray, match_cols: np.ndarray) -> dict:
    ent = nli['entailment_prob']
    con = nli['contradiction_prob']
    neu = nli['neutral_prob']
    labels_idx = nli['labels_idx']
    G, P = ent.shape

    # Matched pairs indices
    matched = list(zip(match_rows.tolist(), match_cols.tolist()))

    # 1-3: rates over matched pairs using argmax label
    ent_count = 0
    con_count = 0
    neu_count = 0
    for i, j in matched:
        idx = int(labels_idx[i, j]) if 0 <= i < G and 0 <= j < P else -1
        if idx == 1:  # entailment
            ent_count += 1
        elif idx == 0:  # contradiction
            con_count += 1
        elif idx == 2:  # neutral
            neu_count += 1
    denom = max(len(matched), 0)
    entailment_rate = (ent_count / denom) if denom > 0 else np.nan
    contradiction_rate = (con_count / denom) if denom > 0 else np.nan
    neutral_rate = (neu_count / denom) if denom > 0 else np.nan

    # 4: GT reason coverage via entailment argmax
    gt_cov_flags = []
    for i in range(G):
        row_labels = labels_idx[i, :]
        # any entailment in this row
        any_ent = np.any(row_labels == 1) if P > 0 else False
        gt_cov_flags.append(1.0 if any_ent else 0.0)
    gt_reason_coverage_entailment = safe_mean(gt_cov_flags) if G > 0 else np.nan

    # 5: Pred reason validity via entailment argmax
    pred_valid_flags = []
    for j in range(P):
        col_labels = labels_idx[:, j]
        any_ent = np.any(col_labels == 1) if G > 0 else False
        pred_valid_flags.append(1.0 if any_ent else 0.0)
    pred_reason_validity_entailment = safe_mean(pred_valid_flags) if P > 0 else np.nan

    # 6: soft alignment mean over matched pairs (ent - con)
    soft_scores_matched = []
    for i, j in matched:
        soft = float(ent[i, j]) - float(con[i, j])
        soft_scores_matched.append(soft)
    nli_soft_alignment_mean = safe_mean(soft_scores_matched)

    # 7: soft coverage mean over GT (max over preds of ent - con)
    gt_soft_best = []
    for i in range(G):
        if P == 0:
            gt_soft_best.append(np.nan)
        else:
            soft_row = (ent[i, :] - con[i, :]).astype(float)
            gt_soft_best.append(np.nanmax(soft_row))
    nli_soft_coverage_mean = safe_mean(gt_soft_best) if G > 0 else np.nan

    # 8: contradiction penalty any
    # For each predicted reason, find the GT pair whose max class prob is highest; if its class is contradiction, flag it.
    contr_flags = []
    for j in range(P):
        if G == 0:
            contr_flags.append(np.nan)
            continue
        # For each GT i, consider the best class probability for pair (i,j)
        # Identify (i*) where max(prob trio) is largest, then check its class argmax
        trio = np.stack([con[:, j], ent[:, j], neu[:, j]], axis=1)  # [G, 3] in order (con, ent, neu)
        best_i = int(np.nanargmax(np.nanmax(trio, axis=1)))
        best_class = int(np.nanargmax(trio[best_i]))
        contr_flags.append(1.0 if best_class == 0 else 0.0)
    contradiction_penalty_any = safe_mean(contr_flags) if P > 0 else np.nan

    return {
        'entailment_rate': entailment_rate,
        'contradiction_rate': contradiction_rate,
        'neutral_rate': neutral_rate,
        'gt_reason_coverage_entailment': gt_reason_coverage_entailment,
        'pred_reason_validity_entailment': pred_reason_validity_entailment,
        'nli_soft_alignment_mean': nli_soft_alignment_mean,
        'nli_soft_coverage_mean': nli_soft_coverage_mean,
        'contradiction_penalty_any': contradiction_penalty_any,
    }


def compute_group2_metrics(reranker: dict, match_rows: np.ndarray, match_cols: np.ndarray, G: int, P: int) -> dict:
    norm = reranker['norm']  # [G, P]

    # Matched average
    matched_scores = [float(norm[i, j]) for i, j in zip(match_rows, match_cols)] if (G > 0 and P > 0) else []
    cross_avg_matched_score = safe_mean(matched_scores)

    # GT coverage: for each GT, max over preds
    if G > 0:
        gt_max = []
        for i in range(G):
            if P == 0:
                gt_max.append(0.0)
            else:
                gt_max.append(float(np.nanmax(norm[i, :])))
        cross_total_gt_coverage = safe_mean(gt_max)
        cross_min_gt_coverage = float(np.nanmin(gt_max)) if len(gt_max) > 0 else np.nan
        cross_recall_like = cross_total_gt_coverage
    else:
        cross_total_gt_coverage = np.nan
        cross_min_gt_coverage = np.nan
        cross_recall_like = np.nan

    # Pred precision: for each pred, max over GT
    if P > 0:
        pred_max = []
        for j in range(P):
            if G == 0:
                pred_max.append(0.0)
            else:
                pred_max.append(float(np.nanmax(norm[:, j])))
        cross_total_pred_precision = safe_mean(pred_max)
        cross_precision_like = cross_total_pred_precision
    else:
        cross_total_pred_precision = np.nan
        cross_precision_like = np.nan

    reason_count_diff = abs(P - G)

    unmatched_gt_penalty = (1.0 - cross_total_gt_coverage) if not np.isnan(cross_total_gt_coverage) else np.nan
    unmatched_pred_penalty = (1.0 - cross_total_pred_precision) if not np.isnan(cross_total_pred_precision) else np.nan

    return {
        'cross_avg_matched_score': cross_avg_matched_score,
        'cross_total_gt_coverage': cross_total_gt_coverage,
        'cross_total_pred_precision': cross_total_pred_precision,
        'cross_min_gt_coverage': cross_min_gt_coverage,
        'cross_recall_like': cross_recall_like,
        'cross_precision_like': cross_precision_like,
        'reason_count_diff': float(reason_count_diff),
        'unmatched_gt_penalty': unmatched_gt_penalty,
        'unmatched_pred_penalty': unmatched_pred_penalty,
    }



## Main Evaluation
- Load dataset and predictions
- Align by id
- Build pairwise matrices, run Hungarian matching
- Compute metrics and collect per-example rows + pairwise reason scores


In [ ]:

# Load and normalize
if COMBINED_INPUT:
    assert PREDICTIONS_PATH.exists(), f"Combined predictions file not found: {PREDICTIONS_PATH}"
    comb_raw = load_json(PREDICTIONS_PATH)
    gt, preds = split_combined_predictions(comb_raw)
else:
    assert DATASET_PATH.exists(), f"Dataset file not found: {DATASET_PATH}"
    assert PREDICTIONS_PATH.exists(), f"Predictions file not found: {PREDICTIONS_PATH} (update PREDICTIONS_PATH above)"
    gt_raw = load_json(DATASET_PATH)
    pred_raw = load_json(PREDICTIONS_PATH)
    gt = normalize_dataset(gt_raw, 'ground_truth')
    preds = normalize_dataset(pred_raw, 'predictions')

gt_map = build_id_map(gt)
pred_map = build_id_map(preds)

common_ids = sorted(set(gt_map.keys()) & set(pred_map.keys()))
print(f"Loaded {len(gt_map)} GT items, {len(pred_map)} predictions, {len(common_ids)} in common.")


# Ensure helper functions exist (in case this cell is run before helper cells)
try:
    compute_verdict_metrics
except NameError:
    def compute_verdict_metrics(gt_verdict: str | None, pred_verdict: str | None, nli: NLIPredictor, rer: ReasoningReranker):
        gt_v = (gt_verdict or '').strip() if isinstance(gt_verdict, str) or gt_verdict is None else str(gt_verdict).strip()
        pd_v = (pred_verdict or '').strip() if isinstance(pred_verdict, str) or pred_verdict is None else str(pred_verdict).strip()
        if not gt_v or not pd_v:
            return {
                'verdict_entailment_rate': np.nan,
                'verdict_nli_soft_coverage_mean': np.nan,
                'verdict_cross_total_gt_coverage': np.nan,
                'verdict_cross_total_pred_precision': np.nan,
            }
        nli_out = nli.predict([(gt_v, pd_v)], batch_size=1)
        probs = nli_out['probs'][0]
        ent_idx = nli.idx['entailment']
        con_idx = nli.idx['contradiction']
        label_idx = int(np.argmax(probs))
        verdict_entailment_rate = 1.0 if label_idx == ent_idx else 0.0
        verdict_soft = float(probs[ent_idx] - probs[con_idx])
        rer_out = rer.score_pairs([(gt_v, pd_v)], batch_size=1)
        norm = float(rer_out['norm'][0]) if rer_out['norm'].shape[0] > 0 else np.nan
        return {
            'verdict_entailment_rate': verdict_entailment_rate,
            'verdict_nli_soft_coverage_mean': verdict_soft,
            'verdict_cross_total_gt_coverage': norm,
            'verdict_cross_total_pred_precision': norm,
        }

try:
    compute_reason_verdict_metrics
except NameError:
    def compute_reason_verdict_metrics(pred_reasons: list[str], pred_verdict: str | None, nli: NLIPredictor, rer: ReasoningReranker):
        pv = (pred_verdict or '').strip() if isinstance(pred_verdict, str) or pred_verdict is None else str(pred_verdict).strip()
        reasons = pred_reasons or []
        reasons = [r.strip() for r in reasons if isinstance(r, str) and r and r.strip()]
        if not pv or len(reasons) == 0:
            return {
                'reason_verdict_entailment_rate': np.nan,
                'reason_verdict_cross_mean': np.nan,
            }
        pairs = [(r, pv) for r in reasons]
        nli_out = nli.predict(pairs, batch_size=max(1, min(32, len(pairs))))
        labels_idx = nli_out['labels_idx']
        ent_idx = nli.idx['entailment']
        entail_count = int((labels_idx == ent_idx).sum())
        entail_rate = (entail_count / len(reasons)) if len(reasons) > 0 else np.nan
        rer_out = rer.score_pairs(pairs, batch_size=max(1, min(32, len(pairs))))
        norm = rer_out['norm']
        cross_mean = float(np.nanmean(norm)) if norm.size > 0 else np.nan
        return {
            'reason_verdict_entailment_rate': float(entail_rate) if entail_rate == entail_rate else np.nan,
            'reason_verdict_cross_mean': cross_mean,
        }

# Initialize models
nli = NLIPredictor(device=DEVICE)
rer = ReasoningReranker(device=DEVICE)

per_example_rows = []
pairwise_rows = []

for ex_id in common_ids:
    g = gt_map[ex_id]
    p = pred_map[ex_id]

    gt_reasons = g.get('reasons', [])
    pred_reasons = p.get('reasons', [])
    G, P = len(gt_reasons), len(pred_reasons)

    # Pairwise matrices
    nli_mat = compute_nli_matrix(gt_reasons, pred_reasons, nli, batch_size=BATCH_SIZE)
    rer_mat = compute_reranker_matrix(gt_reasons, pred_reasons, rer, batch_size=BATCH_SIZE)

    # Matching using hybrid similarity: 0.7*reranker_norm + 0.3*rescaled_nli_soft
    nli_soft = (nli_mat['entailment_prob'] - nli_mat['contradiction_prob']).astype(float)
    nli_soft_rescaled = (nli_soft + 1.0) / 2.0
    hybrid = 0.7 * np.nan_to_num(rer_mat['norm'], nan=0.0) + 0.3 * np.nan_to_num(nli_soft_rescaled, nan=0.0)
    rows, cols, matched_scores = hungarian_match_from_scores(hybrid)

    # Metrics
    g1 = compute_group1_metrics(nli_mat, rows, cols)
    g2 = compute_group2_metrics(rer_mat, rows, cols, G, P)

    # Verdict-level metrics
    v_metrics = compute_verdict_metrics(g.get('verdict'), p.get('verdict'), nli, rer)
    rv_metrics = compute_reason_verdict_metrics(pred_reasons, p.get('verdict'), nli, rer)

    # Select only the 6 main metrics
    selected_g1 = {k: g1.get(k) for k in ['entailment_rate','contradiction_rate','nli_soft_coverage_mean']}
    selected_g2 = {k: g2.get(k) for k in ['cross_total_gt_coverage','cross_total_pred_precision','reason_count_diff']}
    row = {
        'id': ex_id,
        'gt_verdict': g.get('verdict'),
        'pred_verdict': p.get('verdict'),
        'gt_reason_count': G,
        'pred_reason_count': P,
        **selected_g1,
        **selected_g2,
    }
    row.update(v_metrics)
    row.update(rv_metrics)
        # Edge-case handling
    if G == 0 and P == 0:
        row['entailment_rate'] = np.nan
        row['contradiction_rate'] = np.nan
        row['nli_soft_coverage_mean'] = np.nan
        row['cross_total_gt_coverage'] = np.nan
        row['cross_total_pred_precision'] = np.nan
        row['reason_count_diff'] = 0.0
    elif G == 0 and P > 0:
        row['entailment_rate'] = np.nan
        row['contradiction_rate'] = np.nan
        row['nli_soft_coverage_mean'] = np.nan
        row['cross_total_gt_coverage'] = np.nan
        # Pred precision undefined when no GT; set NaN explicitly
        row['cross_total_pred_precision'] = np.nan
        row['reason_count_diff'] = float(P)
    elif G > 0 and P == 0:
        row['entailment_rate'] = np.nan
        row['contradiction_rate'] = np.nan
        # With no predictions, soft coverage is undefined; keep NaN for consistency
        row['nli_soft_coverage_mean'] = np.nan
        # GT coverage with no preds: explicitly 0.0
        row['cross_total_gt_coverage'] = 0.0
        row['cross_total_pred_precision'] = np.nan
        row['reason_count_diff'] = float(G)

    per_example_rows.append(row)

    # Collect pairwise rows for export
    matched_set = set(zip(rows.tolist(), cols.tolist()))
    for i in range(max(G, 1)):
        if i >= G:
            break
        for j in range(max(P, 1)):
            if j >= P:
                break
            pair = {
                'id': ex_id,
                'gt_reason_idx': i,
                'pred_reason_idx': j,
                'gt_reason': gt_reasons[i] if i < G else '',
                'pred_reason': pred_reasons[j] if j < P else '',
                'contradiction_prob': float(nli_mat['contradiction_prob'][i, j]) if G>0 and P>0 else np.nan,
                'neutral_prob': float(nli_mat['neutral_prob'][i, j]) if G>0 and P>0 else np.nan,
                'entailment_prob': float(nli_mat['entailment_prob'][i, j]) if G>0 and P>0 else np.nan,
                'nli_soft_score': (float(nli_mat['entailment_prob'][i, j]) - float(nli_mat['contradiction_prob'][i, j])) if G>0 and P>0 else np.nan,
                'reranker_raw': float(rer_mat['raw'][i, j]) if G>0 and P>0 else np.nan,
                'reranker_norm': float(rer_mat['norm'][i, j]) if G>0 and P>0 else np.nan,
                'matched_flag': 1 if (i, j) in matched_set else 0,
            }
            pairwise_rows.append(pair)

per_example_df = pd.DataFrame(per_example_rows)
pairwise_df = pd.DataFrame(pairwise_rows)

print("Done computing per-example and pairwise tables.")


In [ ]:

# Keep only required columns in a fixed order (removed nli_soft_coverage_mean, verdict_nli_soft_coverage_mean, verdict_cross_total_pred_precision)
required_cols = [
    'id', 'gt_verdict', 'pred_verdict', 'gt_reason_count', 'pred_reason_count',
    'entailment_rate', 'contradiction_rate',
    'cross_total_gt_coverage', 'cross_total_pred_precision', 'reason_count_diff',
    'verdict_entailment_rate', 'verdict_cross_total_gt_coverage',
    'reason_verdict_entailment_rate', 'reason_verdict_cross_mean'
]
# Create any missing columns as NaN to avoid KeyErrors
for col in required_cols:
    if col not in per_example_df.columns:
        per_example_df[col] = np.nan
per_example_df = per_example_df[required_cols]



## Aggregate Summary
Compute dataset-level statistics (mean, std, min, max) over numeric metric columns.


In [ ]:

# Aggregate stats over selected metrics only (dropped nli_soft_coverage_mean, verdict_nli_soft_coverage_mean, verdict_cross_total_pred_precision)
main_metrics = [
    'entailment_rate',
    'contradiction_rate',
    'cross_total_gt_coverage',
    'cross_total_pred_precision',
    'reason_count_diff',
    'verdict_entailment_rate',
    'verdict_cross_total_gt_coverage',
    'reason_verdict_entailment_rate',
    'reason_verdict_cross_mean',
]

if len(per_example_rows) == 0:
    agg_df = pd.DataFrame(columns=['metric_name', 'mean', 'std', 'min', 'max'])
else:
    stats = []
    for col in main_metrics:
        if col not in per_example_df.columns:
            continue
        col_vals = pd.to_numeric(per_example_df[col], errors='coerce')
        stats.append({
            'metric_name': col,
            'mean': float(col_vals.mean(skipna=True)),
            'std': float(col_vals.std(skipna=True)),
            'min': float(col_vals.min(skipna=True)),
            'max': float(col_vals.max(skipna=True)),
        })
    agg_df = pd.DataFrame(stats)

print("Aggregates computed for selected metrics only.")



## Final Outputs
- Preview tables
- CSV-style blocks (plain text via `.to_csv(index=False)`) for copy-paste


In [ ]:

# Preview
print("Per-example metrics preview:")
display(per_example_df)

print("Aggregate summary preview:")
display(agg_df)

print("Pairwise reason scores preview:")
display(pairwise_df.head(50))

# CSV blocks
print("===== per_example_metrics (CSV) =====")
print(per_example_df.to_csv(index=False))

print("===== aggregate_summary (CSV) =====")
print(agg_df.to_csv(index=False))

print("===== pairwise_reason_scores (CSV) =====")
print(pairwise_df.to_csv(index=False))

In [ ]:

# Predicted reasons -> predicted verdict coherence metrics

def compute_reason_verdict_metrics(pred_reasons: list[str], pred_verdict: str | None, nli: NLIPredictor, rer: ReasoningReranker):
    # Normalize inputs
    pv = (pred_verdict or '').strip() if isinstance(pred_verdict, str) or pred_verdict is None else str(pred_verdict).strip()
    reasons = pred_reasons or []
    reasons = [r.strip() for r in reasons if isinstance(r, str) and r and r.strip()]

    if not pv or len(reasons) == 0:
        return {
            'reason_verdict_entailment_rate': np.nan,
            'reason_verdict_cross_mean': np.nan,
        }

    # Build pairs: each reason as premise, verdict as hypothesis/text_b
    pairs = [(r, pv) for r in reasons]

    # NLI entailment rate
    nli_out = nli.predict(pairs, batch_size=max(1, min(32, len(pairs))))
    labels_idx = nli_out['labels_idx']
    ent_idx = nli.idx['entailment']
    entail_count = int((labels_idx == ent_idx).sum())
    entail_rate = (entail_count / len(reasons)) if len(reasons) > 0 else np.nan

    # Reranker mean norm
    rer_out = rer.score_pairs(pairs, batch_size=max(1, min(32, len(pairs))))
    norm = rer_out['norm']
    cross_mean = float(np.nanmean(norm)) if norm.size > 0 else np.nan

    return {
        'reason_verdict_entailment_rate': float(entail_rate) if entail_rate == entail_rate else np.nan,
        'reason_verdict_cross_mean': cross_mean,
    }
